    # Trabajo final · Data Engineer · AeroCuyo

    **Aerolínea regional** · Empresa ficticia. Los datos son sintéticos y ningún resultado representa una situación real de ARCA. No hay credenciales ni infraestructura real.

    **Lo propio de la empresa** (la base operacional y los eventos de sus sistemas) **se genera por código en este notebook**. **Lo externo** (los datos que mandan terceros) **llega como archivos CSV y XLSX** en la carpeta `datos_externos/`, junto a este notebook.

    ## El caso

    AeroCuyo opera 30 vuelos por día entre seis aeropuertos. Dos sistemas aeroportuarios informan los eventos de cada vuelo con husos horarios distintos y, a veces, con horas de retraso. Operaciones quiere la puntualidad cada mañana y alertas de demora en minutos.

    ## Qué tiene que resolver la plataforma

    - Puntualidad por vuelo cada mañana.
- Alerta de demora durante el día.
- Trazabilidad del equipaje.

    ## Restricciones del caso

    - Dos sistemas con husos horarios distintos.
- Eventos tardíos y duplicados.
- Datos de pasajeros: nombre y pasaporte.
- Los archivos externos llegan en CSV y XLSX, y cambian de formato sin aviso.

    Todo lo que el caso **no declara** es una suposición: escribila y justificala.

    Las instrucciones completas (qué se entrega, rúbrica y defensa) están en el PDF **«Trabajo final · Data Engineer · Instrucciones»**. Este notebook es tu lugar de trabajo: vas del **SQL al análisis** sin salir de acá. Elegí la herramienta **después** de fijar el requisito.

## Paso 0 · Instalación

Corré esta celda primero. Si falta alguna librería (incluida `openpyxl`, para leer XLSX), la instala en el mismo Python que ejecuta este notebook.

In [ ]:
# Instalación autocontenida: instala, en el mismo Python que ejecuta este notebook, lo que falte.
import importlib, importlib.util, subprocess, sys

REQUISITOS = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "openpyxl": "openpyxl"}
print(f"Kernel en uso: {sys.executable} (Python {sys.version.split()[0]})")
faltan = [pip for modulo, pip in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    base = [sys.executable, "-m", "pip", "install", "-q"]
    try:
        try:
            subprocess.check_call(base + faltan)
        except subprocess.CalledProcessError:       # el Python del sistema suele rechazar instalaciones
            print("Reintentando con --break-system-packages ...")
            subprocess.check_call(base + ["--break-system-packages"] + faltan)
    except Exception:
        raise SystemExit("No se pudo instalar en este kernel. Elegí el kernel .venv (arriba a la derecha) y volvé a correr esta celda.")
    importlib.invalidate_caches()
    print("Listo.")
else:
    print("Todo instalado: podés seguir.")

## 1 · Las fuentes

La primera celda **genera lo propio** con código comprimido a propósito: no hace falta leerla, y leerla te quita la mitad del trabajo. La segunda lo escribe en la carpeta `fuentes/` (una base SQLite y los eventos en JSON), lista los archivos externos y comprueba que tu base coincide con la que se usó para armarlos.

Te deja estas herramientas: `sql(...)`, `ls()`, `externos()`, `hojas(...)`, `leer_externo(...)`, `leer_eventos(...)`, `dwh()`, `medir(...)`, `regla(...)`, `correr_dag(...)`, `zonas()` y `enmascarar(...)`.

In [ ]:
import zlib, base64
_BLOB = (
    "eNqlV+tu2zYU/p+n4AgYkVpbkZ04aZy5QJK2648WK5qkQGsYBmPSjlpJ1EjJTWv4qfYIe7F9h5JtxVFbYBNyEcnDc/nOVVGS"
    "aZOztEiyb0xYlmZtlolU4hU/mdyTasYmMo0802apP2BG5YVJ2cjmxovS3Lv3fTbTht2zKGUmwJaaK2O9Xlg+bXb0bP2W+uO9"
    "imGReXLWZmAKgVYpuWWdyWCq06nIvRGRyFlgRZLFyomLVYqLPnvCMh/XoalOJjYXuRo6LuM2i+apNmoSpVLdD69NofxSqFT9"
    "iVBGT4tv2iPiYb/vD/YYHsOGsDwo2QUgFkWcT0w6d3T+GRMZKJb87YtPfMD4eaJMNBUH52au0jxKxcFbhZvfBW/j8OW7ZpqL"
    "QqXaTs4joywRXv75vpnwUhupbx2zqzfnzTRXIs4dxcX7y19Q3Fy9bqa4sXeFiARfnbGplhYWxpHNPZHB4EXRZmoBhBXtjwBr"
    "+XvGvmAdOtTI7ZLcDo9JeGAC/ObK472w1+90w07YJfGb1Qmv0F7fnbiQcXcOw9oZPRqOV5BkgumdjqbKIw3brEcaZbGYquEr"
    "EVvlkz5Ph6x7xqyIQS/ZU1LnOkqUVADAu9OFsUMKnVpw9sGp5yOAkigtcrU+r0SNEKpdkBxS8PbHvg8psqAY2WFzQoTPQjqP"
    "5ALnM/7hZvllEB7JFQeoIp2q2BlRRpbns99ZEB4+sHRRBCLLED+eByZt8Di/XNaEdClzTk9P/RXQdLC0ydYACTjLYabHWx87"
    "raTTkqz1etB6y2GWR2DsALE2FZb4/k9vz/ibD50fqnACex8YIFVSQYMUmsZRBohiTUmYiNjrBfBZNwjBl+AMcZtFsx1IDkOm"
    "4M0qrtYPyEoIBwjFhyDx8kBMI51yhwfY7+hFHB8Gldvd5SSVzdS8UCWbH6KmEp+wT3aEVHrWzHkOD3f7DSojQ4yJvovPvxaF"
    "I3LTTwQS0XPWKAdH2og1Ko85UOrZbertpga8dPwMMd8Anc3h6DX0UnBSxKuiHObvOLXvl071uLhFRfvn75I+1Tn74Z3Tw82l"
    "VE/snf7qLtXuuNMNxwZwULE2gMz4+6sltQxs+oPwWFIAV1n2Tli4wmj2iAAZeH6x3MGlW/WwU/f4fkm4tOzggB3DX93Vcv/8"
    "4vLFy1f7I8ta7HhMFIAMZLEWLjmMLqBVPTu6yIw24r8PX3dQkSp7FgWARnC8ELl4ZQTSlOrxVMdFktrhiEdysihUrKm+onXD"
    "CnrTJkJp52VMo8K7TQRBJMUkM3pOyzhWc1FbJyJHVyhi1ymmIkNGSSH5GCWtLP0PtMDWjhrYUWbhbteVyipwJ6lObo1ab9Gs"
    "4RbCRmhCjhSqClleMgr57GRbNCKVCPug69a660W9gdYbZa0lXtSbHzW5Eh+wRMm1Ki8nBG+rtx+UFKj2JX4/I61IztAzU9ch"
    "Ny3RxVceZajVucs/yjW12OaTyCaKukUpbQT6MQU5XWG/DR8UijLcK1mO0mGDy2uERo4btvVshu3OIbEqJRAvsr9k0tm2nZwY"
    "5A094Nr1gEHr6hN3VdqJGhKUVVbmze0Vsht7yoZfJzwchGHJ1DXu/8aPrfnxbd4jaMgaMPIay+luHvfKkdRv6kPhUanXA0ZW"
    "YRyVjxmhFhAX34XAuuQstzEyKCOBq4UL9UEVEzy39I5E4t91Kig6b64veRPcPNZTEfNNTZ/AJtBjQYldBgDWeMOakuc2kiQI"
    "r79w7qrEr4xd+vuUjfBv9HnsQvhzOcxX8xDVRxyiRK0ncFphBA/Cnr87kCESTYDBcjbDxE50GJ7N9I4SefW/p8aFdSk5oqzc"
    "1jWydTTohmMCTzZYzv1xsHYL1NNf3UyL7nDxx+My/+TJMQYW/Dspi/yioYCvq/dJm1Hl7tIoSR9E2xEStdTMq7oG9xu1XmTK"
    "SPISpulsOApoxgyOT9s0NNCk6eBZEDywdWdI3lGVruILxvfHG3zk7FHFhrH1kp2L+aNarayefJnXavG4ctpoxtVfRZShEk2W"
    "NWD3Wx9bSUvu+6tgahd8THP3zE0lSKIqd8ApyWpe7fOqs+N7zqiUdFtrtdyoMKi0iW/5yg+EtfiW86qdYSySWynY/YB590G1"
    "iSDEgBkeHfuVb7pVB62+JZdc3oLrkjtrKfGokfKqa1mXK3ZFDQnmRgtHQa+btLXutobXRE7nwWer0xibCO3Vam8PBRxH9G1Q"
    "/7zc+xcoo5aG"
)
exec(zlib.decompress(base64.b64decode(_BLOB)).decode("utf-8"))   # define generar(): código comprimido a propósito

In [ ]:
import os, json, sqlite3, time, shutil
pd.options.display.width = 200; pd.options.display.max_columns = 40; pd.options.display.float_format = "{:,.3f}".format
t = generar(); DESTINO = "fuentes"; shutil.rmtree(DESTINO, ignore_errors=True); os.makedirs(DESTINO)   # lo propio de la empresa: base operacional y eventos
_con = sqlite3.connect(f"{DESTINO}/operacional.db")
for nombre, df in t["db"].items(): df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, _con, index=False)
_con.close()
if t["eventos"]: os.makedirs(f"{DESTINO}/eventos")
for nombre, ev in t["eventos"].items():
    with open(f"{DESTINO}/eventos/{nombre}", "w", encoding="utf-8") as f:
        for x in ev: f.write(json.dumps(x, ensure_ascii=False) + "\n")
EXTERNOS = next((c for c in ("datos_externos", "../datos_externos") if os.path.isdir(c)), None)
if EXTERNOS is None: print("⚠ No encuentro la carpeta datos_externos. Abrí este notebook desde la carpeta de tu equipo, junto a datos_externos/.")

def sql(consulta, base=f"{DESTINO}/operacional.db", parametros=None):
    """Corre una consulta SQL sobre la base operacional (o sobre otra base) y devuelve un DataFrame."""
    c = sqlite3.connect(base)
    try: return pd.read_sql_query(consulta, c, params=parametros)
    finally: c.close()
def ls(carpeta=DESTINO):
    """Lista lo propio que se generó (base y eventos) con su tamaño."""
    for raiz, _, nombres in sorted(os.walk(carpeta)):
        for n in sorted(nombres): print(f"{os.path.join(raiz, n):<62} {os.path.getsize(os.path.join(raiz, n)) / 1024:>9,.0f} KB")
def externos():
    """Lista los archivos externos (CSV y XLSX) con su tamaño y, en los XLSX, sus hojas."""
    for n in sorted(os.listdir(EXTERNOS)):
        ruta = os.path.join(EXTERNOS, n); extra = ("  hojas: " + ", ".join(pd.ExcelFile(ruta).sheet_names)) if n.endswith(".xlsx") else ""
        print(f"{n:<40} {os.path.getsize(ruta) / 1024:>8,.0f} KB{extra}")
def hojas(nombre):
    """Nombres de las hojas de un archivo XLSX externo."""
    return pd.ExcelFile(os.path.join(EXTERNOS, nombre)).sheet_names
def leer_externo(nombre, hoja=0, **opciones):
    """Lee un archivo externo (CSV o XLSX) como DataFrame. No arregla nada: ver cómo viene es parte del trabajo."""
    ruta = os.path.join(EXTERNOS, nombre)
    return pd.read_excel(ruta, sheet_name=hoja, **opciones) if nombre.endswith(".xlsx") else pd.read_csv(ruta, **opciones)
def leer_eventos(nombre):
    """Lee un archivo de eventos JSON (uno por línea) como DataFrame."""
    return pd.read_json(f"{DESTINO}/eventos/{nombre}", lines=True)
def dwh(archivo="dwh.db"):
    """Conexión al almacén que vas a modelar (un archivo SQLite nuevo)."""
    return sqlite3.connect(archivo)
def medir(funcion, *args, **kwargs):
    """Mide cuánto tarda una función y devuelve (resultado, segundos)."""
    t0 = time.perf_counter(); r = funcion(*args, **kwargs); return r, time.perf_counter() - t0
resultados_calidad = []
def regla(nombre, valor, umbral, mayor_es_mejor=False, descripcion=""):
    """Registra una regla de calidad medible: compara un valor con un umbral y deja el resultado."""
    ok = valor >= umbral if mayor_es_mejor else valor <= umbral
    resultados_calidad.append({"regla": nombre, "valor": valor, "umbral": umbral, "cumple": ok, "descripción": descripcion}); print(("✔" if ok else "✘"), nombre, "→", valor, f"(umbral {umbral})"); return ok
def correr_dag(tareas, dependencias, reintentos=2):
    """Ejecuta tareas (nombre → función) respetando dependencias, con reintentos. Si una falla, sus dependientes no corren."""
    estado = {}; pendientes = list(tareas)
    while pendientes:
        listas = [n for n in pendientes if all(estado.get(d) == "ok" for d in dependencias.get(n, []))]; bloq = [n for n in pendientes if any(estado.get(d) in ("falló", "omitida") for d in dependencias.get(n, []))]
        for n in bloq: estado[n] = "omitida"; pendientes.remove(n); print(f"  ⏭  {n}: omitida (falló una dependencia)")
        if not listas and not bloq: raise RuntimeError(f"Dependencias circulares o faltantes: {pendientes}")
        for n in listas:
            pendientes.remove(n)
            for intento in range(1, reintentos + 2):
                try: tareas[n](); estado[n] = "ok"; print(f"  ✔ {n}" + (f" (intento {intento})" if intento > 1 else "")); break
                except Exception as ex: print(f"  ✘ {n}, intento {intento}: {type(ex).__name__}: {ex}"); estado[n] = "falló"
    return estado
def zonas(base="lake"):
    """Crea las carpetas de las zonas de datos."""
    for z in ("cruda", "curada", "variables", "consumo"): os.makedirs(f"{base}/{z}", exist_ok=True)
    return sorted(os.listdir(base))
def enmascarar(valor, visibles=2):
    """Enmascara un texto dejando visibles los últimos caracteres."""
    s = str(valor); return "*" * max(len(s) - visibles, 0) + s[-visibles:]

HUELLA = {'vuelos': [210, 14700.0], 'reservas': [12237, 778195900.0], 'operativos.jsonl': 433}
def comprobar_base():
    """Comprueba que la base que generaste coincide con la que usaron para armar los archivos externos."""
    bien = True
    for nombre, df in t["db"].items():
        filas, suma = len(df), round(float(df.select_dtypes("number").sum().sum()), 2); e_filas, e_suma = HUELLA[nombre]
        if filas != e_filas or abs(suma - e_suma) > max(0.01, abs(e_suma) * 1e-6): bien = False; print(f"⚠ {nombre}: no coincide ({filas} filas)")
    for nombre, ev in t["eventos"].items():
        if len(ev) != HUELLA[nombre]: bien = False; print(f"⚠ {nombre}: no coincide ({len(ev)} eventos)")
    print("✔ Tu base coincide con los archivos externos." if bien else "⚠ Tu base no coincide con los archivos externos: reiniciá el kernel y volvé a correr todo. Si sigue, avisá al facilitador.")
    return bien
print("Lo propio, generado en la carpeta", DESTINO + "/")
ls(); print("\nLo externo, en la carpeta", EXTERNOS + "/")
externos(); print(); comprobar_base()

Los archivos externos **cambian de formato sin aviso**, y los datos propios traen duplicados, datos tardíos y datos personales. Descubrirlo, decidir qué hacer con cada caso y dejarlo escrito es parte de la evidencia. `leer_externo(...)` lee el archivo tal como viene: no arregla nada.

In [ ]:
# Explorá lo propio y lo externo
sql("SELECT name FROM sqlite_master WHERE type = 'table'")

## Qué tiene que recorrer tu trabajo

| Módulo | Qué se entrega |
| --- | --- |
| 1 · Arquitectura moderna | Un diagrama con fuentes, almacenamiento, procesamiento y consumo, y la justificación de la elección (warehouse, lake o lakehouse) con **una alternativa descartada**. |
| 2 · Modelado | Modelo conceptual, lógico y físico del dominio y un modelo dimensional: un hecho de vuelo y otro de eventos de vuelo, con las dimensiones aeropuerto, aeronave, tiempo y pasajero. Con su DDL. |
| 3 · PL/SQL | Un procedimiento (o un conjunto de sentencias SQL) que calcule la puntualidad por vuelo reconciliando los eventos de los dos sistemas, con manejo de errores y una medición de tiempos antes y después de optimizarlo. |
| 4 · Ingestión (NiFi) | El flujo para ingerir la base de vuelos y reservas, los eventos operativos, el equipaje, la meteorología de los aeropuertos y los slots asignados (XLSX): procesadores, conexiones, manejo de errores y monitoreo. Se implementa en NiFi o, si no hay entorno, como diagrama más un script equivalente que cargue los datos entregados. |
| 5 · Orquestación (Airflow) | Un DAG con dependencias, calendario (una vez por día para la puntualidad y de forma continua para las alertas), reintentos e idempotencia, y la decisión sobre los datos que llegan tarde. |
| 6 · Governance | Al menos cinco reglas de calidad **medibles**, aplicadas a los datos entregados, con sus resultados; roles (dueño y responsable) y un contrato de datos. |
| 7 · MLOps | Un modelo de predicción de demora: si lo incluyen, ficha de versión, criterios de promoción, monitoreo y marcha atrás. |
| 8 · Plataformas | Las zonas de datos (cruda, curada, variables y consumo) con sus reglas de paso; si hay un modelo, la arquitectura para Machine Learning. |
| 9 · Seguridad y metadatos | Clasificación de los datos (nombre y pasaporte de los pasajeros), enmascaramiento y accesos por rol; un catálogo mínimo con el linaje de un indicador. |
| 10 · Proyecto | Una matriz de riesgos, un backlog priorizado y una defensa de 8 minutos. |

**Sobre el modelo (opcional):** Un modelo de predicción de demora: si lo incluyen, ficha de versión, criterios de promoción, monitoreo y marcha atrás.

## 1 · Arquitectura moderna

Escribí la arquitectura (fuentes propias y externas, almacenamiento, procesamiento, consumo) y la justificación, con **una alternativa descartada**. Podés usar un diagrama de texto o Mermaid.

| Campo | Contenido |
| --- | --- |
| Decisión |  |
| Restricción comprobable |  |
| Opción elegida y costo que se acepta |  |
| **Alternativa descartada y por qué** |  |
| Control que detecta el fallo y qué se hace cuando suena |  |

## 2 · Modelado

Modelo conceptual, lógico y físico, y un modelo dimensional: un hecho de vuelo y otro de eventos de vuelo, con las dimensiones aeropuerto, aeronave, tiempo y pasajero. Escribí el DDL y creá las tablas en tu almacén.

In [ ]:
almacen = dwh()
almacen.executescript("""
-- Tu DDL
""")

## 3 · PL/SQL

Un procedimiento que calcule la puntualidad por vuelo reconciliando los eventos de los dos sistemas. SQLite no tiene PL/SQL: escribilo como sentencias SQL ejecutadas desde una función de Python con manejo de errores, y medí los tiempos antes y después de optimizarlo con `medir(...)`.

In [ ]:
def cargar():
    # Tu carga: SQL + manejo de errores
    pass

# _, segundos = medir(cargar)

## 4 · Ingestión

El flujo para ingerir la base de vuelos y reservas, los eventos operativos, el equipaje, la meteorología de los aeropuertos y los slots asignados (XLSX). Si tenés Apache NiFi, diseñalo ahí; si no, escribí el diagrama (procesadores, conexiones, manejo de errores y monitoreo) y un script equivalente. Cada archivo externo es un origen distinto, con su propio formato y sus propios problemas.

In [ ]:
# Tu ingestión
externos()

## 5 · Orquestación

Un DAG con dependencias, calendario (una vez por día para la puntualidad y de forma continua para las alertas), reintentos e idempotencia, y la decisión sobre los datos que llegan tarde. Con `correr_dag(tareas, dependencias)` lo ejecutás en este notebook.

In [ ]:
def extraer(): pass
def transformar(): pass
def cargar_dwh(): pass

# estados = correr_dag({"extraer": extraer, "transformar": transformar, "cargar": cargar_dwh},
#                      {"transformar": ["extraer"], "cargar": ["transformar"]})

## 6 · Governance

Al menos **cinco reglas de calidad medibles** aplicadas a los datos propios y externos, con sus resultados, más los roles (dueño y responsable) y un contrato de datos. «No hay nulos» no alcanza como definición de calidad.

In [ ]:
# regla("nombre", valor_medido, umbral)

In [ ]:
pd.DataFrame(resultados_calidad)

## 7 · MLOps

Un modelo de predicción de demora: si lo incluyen, ficha de versión, criterios de promoción, monitoreo y marcha atrás.

In [ ]:
# Ficha de versión, criterios de promoción, monitoreo y marcha atrás (si corresponde)

## 8 · Plataformas

Las zonas de datos (cruda, curada, variables y consumo) con sus reglas de paso. Si hay un modelo, la arquitectura para Machine Learning.

In [ ]:
zonas()

## 9 · Seguridad y metadatos

Clasificación de los datos (nombre y pasaporte de los pasajeros), enmascaramiento y accesos por rol, y un catálogo mínimo con el linaje de un indicador.

In [ ]:
clasificacion = {
    # "tabla.columna": "público | interno | confidencial | sensible",
}
catalogo = {
    # "indicador": {"dueño": "", "responsable": "", "definición": "", "frescura": "", "linaje": []},
}

## 10 · Proyecto: riesgos, backlog y defensa

**Matriz de riesgos**

| Riesgo | Probabilidad | Impacto | Cómo se detecta | Cómo se contiene y se recupera | Responsable |
| --- | --- | --- | --- | --- | --- |
|  |  |  |  |  |  |

**Backlog priorizado**

| Prioridad | Qué | Por qué ahora | Esfuerzo | Depende de |
| --- | --- | --- | --- | --- |
|  |  |  |  |  |

**Guion de la defensa de 8 minutos:** problema y decisión (1) · diseño (3) · evidencia (2) · fallas y recuperación (1) · lo que queda abierto (1).

### Antes de entregar

- [ ] Reinicié el kernel y corrí todo el notebook de punta a punta.
- [ ] Escribí qué suponemos y el caso no declara.
- [ ] Elegimos la herramienta después de fijar el requisito.
- [ ] Cada decisión tiene una alternativa descartada y el costo que aceptamos.
- [ ] Encontramos los problemas de las fuentes propias y externas y dijimos qué hacemos con cada uno.
- [ ] Cada control dice **qué se hace cuando falla** y quién decide.
- [ ] La carga es idempotente y se puede reprocesar un día cualquiera.
- [ ] Los datos sensibles están clasificados, enmascarados y con acceso por rol.
- [ ] Si el caso lleva un modelo, no usa información posterior al evento.
- [ ] Ensayamos la defensa en 8 minutos.